In [ ]:
# Install dependencies if needed
# %pip install -r ../requirements.txt
# dbutils.library.restartPython()

In [ ]:
import sys
import time
from pprint import pprint

sys.path.insert(0, '..')

from multiAgentSystem.agents.critic import critic_node
from experiments.mlflow_setup import (
    setup_experiment,
    create_agent_run,
    log_agent_metrics,
    log_state_snapshot,
    enable_autologging
)
from experiments.mock_data import CRITIC_TEST_STATES, get_test_state

print("✓ Imports successful")

In [ ]:
# Setup MLflow experiment
enable_autologging()
experiment_id = setup_experiment("critic")
print(f"Experiment ID: {experiment_id}")

In [ ]:
def run_critic_test(scenario_name: str, verbose: bool = True):
    """
    Run a single critic test scenario with MLflow tracking.
    """
    scenario = get_test_state("critic", scenario_name)
    test_state = scenario["state"].copy()
    expected = scenario["expected"]
    
    with create_agent_run("critic", scenario=scenario_name) as run:
        log_state_snapshot(test_state, prefix="input")
        
        start_time = time.time()
        try:
            result = critic_node(test_state)
            success = True
        except Exception as e:
            result = {"error": str(e)}
            success = False
        latency_ms = (time.time() - start_time) * 1000
        
        # Extract critic output fields (matching state definition)
        confidence = result.get("confidence", 0.0)
        critique = result.get("critique", "")
        critic_approved = result.get("critic_approved", False)
        
        log_agent_metrics(
            latency_ms=latency_ms,
            success=success,
            additional_metrics={
                "confidence": confidence,
                "critic_approved": 1.0 if critic_approved else 0.0,
                "critique_length": len(critique),
            }
        )
        
        log_state_snapshot(result, prefix="output")
        
        # Check expected outcomes
        passed = True
        
        # Check confidence range
        min_conf = expected.get("min_confidence", 0.0)
        max_conf = expected.get("max_confidence", 1.0)
        if not (min_conf <= confidence <= max_conf):
            passed = False
        
        # Check approval status if expected
        if "should_approve" in expected:
            if expected["should_approve"] != critic_approved:
                passed = False
        
        # Check confidence adjustment
        input_conf = test_state.get("confidence", 0.5)
        if expected.get("confidence_lowered") and confidence >= input_conf:
            passed = False
        if expected.get("confidence_stable") and abs(confidence - input_conf) > 0.1:
            passed = False
            
        # Check critique if expected
        if expected.get("reason_mentions_confidence") and "confidence" not in critique.lower():
            passed = False
        
        import mlflow
        mlflow.log_metric("test_passed", 1.0 if passed else 0.0)
        
        if verbose:
            status = "✅ PASSED" if passed else "❌ FAILED"
            print(f"\n{status} - {scenario_name}")
            print(f"  Description: {scenario['description']}")
            print(f"  Latency: {latency_ms:.2f}ms")
            print(f"  Input confidence: {input_conf:.2f}")
            print(f"  Output confidence: {confidence:.2f} (expected: {min_conf:.2f}-{max_conf:.2f})")
            print(f"  Approved: {critic_approved}")
            if critique:
                print(f"  Critique preview: {critique[:200]}...")
        
        return result, passed, latency_ms

## Test 1: Well-Supported Draft

A draft with strong evidence support should receive high confidence and not be sent back for revision.

In [ ]:
result_1, passed_1, latency_1 = run_critic_test("well_supported_draft")

## Test 2: Unsupported Claims

A draft making claims without evidence should receive low confidence and be flagged for revision.

In [ ]:
result_2, passed_2, latency_2 = run_critic_test("unsupported_claims")

## Test 3: Low Confidence Draft

A draft with minimal evidence should be flagged for revision with specific feedback on missing areas.

In [ ]:
result_3, passed_3, latency_3 = run_critic_test("low_confidence_draft")

## Summary

In [ ]:
print("=" * 60)
print("CRITIC AGENT TEST SUMMARY")
print("=" * 60)

tests = [
    ("well_supported_draft", passed_1, latency_1),
    ("unsupported_claims", passed_2, latency_2),
    ("low_confidence_draft", passed_3, latency_3),
]

total_passed = sum(1 for _, passed, _ in tests if passed)
avg_latency = sum(lat for _, _, lat in tests) / len(tests)

for name, passed, latency in tests:
    status = "✅" if passed else "❌"
    print(f"  {status} {name}: {latency:.2f}ms")

print("=" * 60)
print(f"Total: {total_passed}/{len(tests)} passed")
print(f"Average latency: {avg_latency:.2f}ms")
print("=" * 60)